# Dat A Project RQ3 EDA


---

## Research Question 3 EDA

To what extent can patient demographics, ADHD medication class, dosage, administration route, and concurrent medication use predict the reporting of serious outcomes like hospitalization, life-threatening events, or death, and how do traditional statistical models compare with machine-learning approaches in predictive performance?

---

## ADHD study data: Overview

I am using the parsed down dataset `faers_adhd_study`, which holds only the project's ADHD medication study group: 94,803 FDA adverse event cases (Q4 2012 to Q2 2026) where a stimulant or nonstimulant ADHD drug is a suspect.

| Table | One row per |
|---|---|
| `adhd_cases` | case: class, dates, age, sex, reporter, every outcome flag, death / life-threatening / hospitalization |
| `adhd_drug` | drug listed on the case (ADHD and other drugs) |
| `adhd_reac` | reaction term (MedDRA PT) |
| `adhd_outc`, `adhd_ther`, `adhd_indi`, `adhd_rpsr` | outcome code, therapy dates, indication, report source |

All tables join on `caseid`. Field meanings: `FAERS study data dictionary.xlsx`.

---

## RQ3 Predictor EDA Summary

Predictors from `adhd_cases` and `adhd_drug` were inspected for missingness, data quality, and serious-outcome rate. Serious = death, life-threatening event, or hospitalization (35,162 current-era cases; 7,702 serious, 21.9%).

### Predictor inspection

| Variable | Missing / Unknown | Key finding | Serious-outcome rate | Decision |
|---|---|---|---|---|
| `rq1_group` | 0% | Heavily imbalanced: STIMULANT 88.6%, NONSTIMULANT 10.2%, MULTI_ADHD_CLASS 1.2% (405 cases, 116 serious) | STIMULANT 21.1%, NONSTIMULANT 27.8%, MULTI 28.6% | ✅ Keep |
| `sex_clean` | 28.1% (None + UNKNOWN combined) | F and M balanced; unknown/missing overlaps strongly with missing age (92.2%) | F 27.1%, M 27.4%, Unknown/Missing 8.3% | ✅ Keep |
| `age_years` | ~51% | One value >120 recoded as missing; ages 0-120 retained; missingness is systematic, not random | Reported 36.7% vs. missing 7.7% | 🟡 Keep, handle missingness explicitly |
| `wt_kg_clean` | ~81% | Values of 0 and 849 kg recoded as missing; available-weight cases are not representative | Reported 19.3% vs. missing 22.5% | 🔴 Exclude from primary model; consider sensitivity analysis |
| `dose_structured` | n/a (31.5% reported) | Built from the first suspect ADHD drug row; lower when sex unknown (21.1%) or age missing (28.3%) | Dose reported 19.5% vs. not 23.0% | ✅ Keep |
| `route_group` | 67.2% unknown/missing | Oral 26.7%, Transdermal 3.4%, Other 2.6%; Transdermal is almost entirely stimulant cases | Oral 30.5%, Other 30.9%, Unknown 19.2%, Transdermal 1.4% (17 of 1,203) | ✅ Keep |
| `other_meds_group` | n/a (60.0% list none) | Long tail (max 1,456) handled by grouping 0 / 1-2 / 3-5 / 6+; unknown-sex cases mostly list none (84.1%) | 0: 8.9%, 1-2: 30.1%, 3-5: 44.2%, 6+: 57.5% | ✅ Keep |

### Timeline check

Cases were placed on the timeline using `init_month`, the month the FDA first received the case. No cases were missing a date (range: April 2007 to June 2026). Periods follow the project plan: train on cases received 2022-2024, test on January 2025-February 2026.

| Period | Cases | % of cases | Serious cases | Serious-outcome rate |
|---|---:|---:|---:|---:|
| Train (2022-2024) | 22,059 | 62.7% | 4,571 | 20.7% |
| Test (Jan 2025-Feb 2026) | 9,356 | 26.6% | 2,342 | 25.0% |
| Before 2022 | 996 | 2.8% | 333 | 33.4% |
| After Feb 2026 | 2,751 | 7.8% | 456 | 16.6% |
| **Total** | **35,162** | **100%** | **7,702** | **21.9%** |

- Both the train and test periods have enough cases and serious outcomes for modeling.
- The serious-outcome rate is higher in the test period (25.0%) than in the training period (20.7%). This should be kept in mind when interpreting model performance and calibration.
- 3,747 cases (10.7%) were first received before 2022 or after February 2026, so they fall outside both periods. They are excluded from model training and testing, leaving **31,415 cases (89.3%)** for modeling. They remain in `cases` and are identified by the `split` column.
- A case first received before 2022 can be in the current-era database because a later follow-up report was counted.

### Leakage check

Each planned predictor was checked for a group with a serious-outcome rate near 0% or 100%, which would suggest the field gives away the outcome.

| Predictor | Serious-rate range across groups | Near 0% or 100%? |
|---|---|---|
| `rq1_group` | 21.1% to 28.6% | No |
| `sex_clean` | 8.3% to 27.4% | No |
| `age_years` (reported vs. missing) | 7.7% to 36.7% | No |
| `dose_structured` | 19.5% to 23.0% | No |
| `route_group` | 1.4% to 30.9% | Transdermal is low, but it is a small group (17 of 1,203) |
| `other_meds_group` | 8.9% to 57.5% | No |

No predictor was removed for leakage because none of them had a serious-outcome rate near 0% or 100%.

### Notes
- Route and dose were taken from one drug row per report: the first suspect (primary or secondary) ADHD medication, with the primary suspect first and then the lowest drug sequence number.
- Missing demographic information (sex, age, weight) tends to occur together, and reports with less information have lower serious-outcome rates.
- All differences are descriptive and do not indicate that any variable causes a serious outcome.

---

## 1. Setup & Connect to DB

1. `pip install duckdb==1.4.5 pandas matplotlib jupyterlab`
2. MotherDuck token (ask the team lead; it is not stored in this repository)
3. Use `faers_adhd_study`

In [ ]:
%pip install duckdb==1.4.5 pandas matplotlib jupyterlab

In [ ]:
import os
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
import getpass
os.environ["motherduck_token"] = getpass.getpass(
"Paste your MotherDuck token: "
)

Paste your MotherDuck token: ··········


In [ ]:
con = duckdb.connect(
    "md:",
    config={
        "motherduck_token": os.environ["motherduck_token"]
    }
)

In [ ]:
con.sql("SHOW DATABASES").show()

┌───────────────────────────────┐
│         database_name         │
│            varchar            │
├───────────────────────────────┤
│ TCAD_Data                     │
│ capstone_faers_study_combined │
│ faers_2012q4_2021q4           │
│ faers_2014q1_2021q4           │
│ faers_2022q1_2026q2           │
│ faers_adhd_study              │
│ faers_study_2012q4_2021q4     │
│ faers_study_2014q1_2021q4     │
│ faers_study_2022q1_2026q2     │
│ faers_study_combined          │
│ md_information_schema         │
├───────────────────────────────┤
│            11 rows            │
└───────────────────────────────┘



In [ ]:
con.sql("USE faers_adhd_study")


In [ ]:
con.sql("SHOW TABLES").show()

┌─────────────┐
│    name     │
│   varchar   │
├─────────────┤
│ adhd_cases  │
│ adhd_drug   │
│ adhd_indi   │
│ adhd_outc   │
│ adhd_reac   │
│ adhd_rpsr   │
│ adhd_ther   │
│ study_build │
└─────────────┘



---

## Narrow the adhd_cases table to 2022-2026 data
Report format is consistent, ensuring that each predictor means the same thing across cases.

In [ ]:
cases = con.sql("""
    SELECT * FROM adhd_cases
    WHERE era = 'current'
""").df()

print(len(cases), cases["caseid"].nunique())

35162 35162


---

## Create the Prediction Column
`serious` with binary values `true` or `false` indicates if the case has a death, life-threatening event or hospitalization event documented.

In [ ]:
flags = ["death", "life_threatening", "hospitalization"]
cases["serious"] = cases[flags].fillna(False).any(axis=1)

# How many cases are 'serious' vs. not?
print(cases["serious"].value_counts())

# What proportion of the cases are 'serious' vs. not?
print(cases["serious"].value_counts(normalize=True))

serious
False    27460
True      7702
Name: count, dtype: int64
serious
False    0.780957
True     0.219043
Name: proportion, dtype: float64


Roughly 78% of cases do not have a death, life-threatening event or hospitalization event documented.

---

## Inspect `adhd_cases` Table
This is the base table we will use to answer RQ3. Before joining any other tables to it, we will want to check for missing values and differences in the rate of occurrence of these death, life-threatening or hospitalization events between groups.

In [ ]:
# How much is missing?
print(cases[["age_years", "sex_category", "wt_kg", "rq1_group"]].isna().mean())

age_years       0.509812
sex_category    0.238496
wt_kg           0.809055
rq1_group       0.000000
dtype: float64


Missing data is not equal across the RQ3 predictors. Let's inspect each variable more closely to better understand the data.

### Inspect `sex_category` variable

In [ ]:
# Make a clean copy of the sex variable
cases["sex_clean"] = cases["sex_category"].fillna("UNKNOWN")

# Combine None/missing and UNKNOWN into one category
cases["sex_clean"] = cases["sex_clean"].replace("UNKNOWN", "Unknown/Missing")

sex_summary = cases["sex_clean"].value_counts().to_frame("Count")

sex_summary["Percent"] = (
    cases["sex_clean"].value_counts(normalize=True) * 100
).round(1)

sex_summary

,Count,Percent
sex_clean,,
F,13214,37.6
M,12057,34.3
Unknown/Missing,9891,28.1


Sex information is unavailable or unknown for approximately **28% of cases**. Among cases with a reported sex category, the distribution between F and M is relatively balanced.

#### Check serious outcomes by sex category

In [ ]:
sex_outcome_summary = cases.groupby("sex_clean")["serious"].agg(
    cases="count",
    serious_cases="sum",
    serious_rate="mean"
)

sex_outcome_summary["serious_percent"] = (
    sex_outcome_summary["serious_rate"] * 100
).round(1)

sex_outcome_summary

,cases,serious_cases,serious_rate,serious_percent
sex_clean,,,,
F,13214,3575,0.270546,27.1
M,12057,3302,0.273866,27.4
Unknown/Missing,9891,825,0.083409,8.3


F and M have almost identical serious-outcome rates, while the Unknown/Missing group is dramatically different. Only 8.3% of cases with unknown/missing sex had a serious outcome, compared with roughly 27% when sex was reported.

#### `sex_category` Inspection Conclusion
Sex information was reported as F for 37.6% of cases and M for 34.3%, while 28.1% had unknown or missing sex information. Serious-outcome rates were nearly identical for cases reported as F (27.1%) and M (27.4%). In contrast, cases with unknown or missing sex had a substantially lower serious-outcome rate of 8.3%. Unknown/missing sex was also strongly associated with missing age, suggesting that demographic missingness is systematic rather than random.

**Overall Recommendation: `sex_category` will be retained for further analysis, with unknown and missing values combined into a separate category (Unknown/Missing) rather than excluding these cases.**

*Note: The variable that this version of the data is stored in is the `sex_clean` variable.*

### Inspect `age_years` variable

We already know the 51% of `age_years` is missing. Let's inspect the ages that actually exist.

In [ ]:
cases["age_years"].describe()

,age_years
count,17236.000000
mean,29.797313
std,16.321040
min,0.000000
25%,16.000000
50%,29.000000
75%,40.000000
max,300.000000


Outlier of 300 detected in the data. Let's further investigate to see how many other outliers are present. Choose 120 as the 'high' cut-off and 1 as the 'low' cut-off.

In [ ]:
cases[cases["age_years"] > 120]["age_years"].value_counts().sort_index()

,count
age_years,
300.0,1


Only 1 outlier on the high end. Let's remove it and verify.

In [ ]:
cases.loc[cases["age_years"] > 120, "age_years"] = np.nan
cases["age_years"].describe()

,age_years
count,17235.000000
mean,29.781636
std,16.191207
min,0.000000
25%,16.000000
50%,29.000000
75%,40.000000
max,96.000000


Now the max value is 96, which is much more plausible.

In [ ]:
cases[cases["age_years"] < 1]["age_years"].value_counts().sort_index()

,count
age_years,
0.000000,28
0.002738,14
0.005476,2
0.008214,2
0.010951,3
0.013689,3
0.019165,3
0.038330,3
0.046543,1


In [ ]:
cases.loc[
    cases["age_years"] == 0,
    ["age_years", "age_band_fine", "rq1_group", "sex_clean", "serious"]
]

,age_years,age_band_fine,rq1_group,sex_clean,serious
404,0.0,0_5,STIMULANT,M,True
2317,0.0,0_5,STIMULANT,F,False
3314,0.0,0_5,STIMULANT,Unknown/Missing,False
5086,0.0,0_5,STIMULANT,F,False
5587,0.0,0_5,STIMULANT,M,False
5643,0.0,0_5,STIMULANT,M,False
9283,0.0,0_5,STIMULANT,Unknown/Missing,False
9686,0.0,0_5,NONSTIMULANT,F,True
12511,0.0,0_5,STIMULANT,M,False
12938,0.0,0_5,STIMULANT,Unknown/Missing,True


In [ ]:
cases.loc[
    cases["age_years"] == 0,
    "age_band_fine"
].value_counts(dropna=False)

,count
age_band_fine,
0_5,28


Age required limited data cleaning. One implausible age greater than 120 years was identified and recoded as missing rather than removing the entire case. Ages below one year were retained because fractional values were consistent with infant ages, and all 28 records with an age of 0 were categorized in the 0–5 age band. Therefore, ages from 0 through 120 were retained for analysis.

#### Investigate if `age_band_fine` is an alternative variable
Even when exact age is missing, do I still have an age group?

In [ ]:
age_band_summary = cases["age_band_fine"].value_counts(
    dropna=False
).to_frame("Count")

age_band_summary["Percent"] = (
    cases["age_band_fine"]
    .value_counts(dropna=False, normalize=True) * 100
).round(1)

age_band_summary

,Count,Percent
age_band_fine,,
None,17927,51.0
26_44,6864,19.5
45_64,2613,7.4
12_17,2492,7.1
18_25,2356,6.7
6_11,2172,6.2
65_PLUS,453,1.3
0_5,285,0.8


`age_band_fine` does not solve the missing-age problem. About **51%** is missing there too.

#### Investigate if missing age matters
How many cases with serious outcomes have missing `age_years` values?

In [ ]:
cases["age_missing"] = cases["age_years"].isna()

In [ ]:
cases.groupby("age_missing")["serious"].agg(
    cases="count",
    serious_cases="sum",
    serious_rate="mean"
)

,cases,serious_cases,serious_rate
age_missing,,,
False,17235,6330,0.367276
True,17927,1372,0.076533


Cases with a **reported age had a serious-outcome rate of approximately** **36.7%**, compared with only 7.7% among cases where age was missing.

#### Investigate if certain groups are more likely to report age

Let's compare medication groups

In [ ]:
pd.crosstab(
    cases["rq1_group"],
    cases["age_missing"],
    normalize="index"
) * 100

age_missing,False,True
rq1_group,,
MULTI_ADHD_CLASS,70.864198,29.135802
NONSTIMULANT,60.255343,39.744657
STIMULANT,47.432111,52.567889


Yes, there is a difference by medication group. The stimulant group is the only group where more than half of cases are missing age.

This is important because one of the thing we want to use to predict serious outcomes is medication group. If we simply deleted every case with a missing age we would disproportionately remove stimulant cases.

**Recommendation: Do not drop cases simply because age is missing.**

Are certain sex indications more likely to report age?

In [ ]:
pd.crosstab(
    cases["sex_clean"],
    cases["age_missing"],
    normalize="index"
) * 100

age_missing,False,True
sex_clean,,
F,63.917058,36.082942
M,66.484200,33.515800
Unknown/Missing,7.815186,92.184814


This is another interesting result. When sex is unknown or missing, ~92% of these cases also have age missing. This suggests age and sex missingness are strongly connected.

#### `age_years` Inspection Conclusion
Age was missing for approximately 51% of cases. Missingness was not evenly distributed across the dataset. Age was missing in 52.6% of stimulant cases, compared with 39.7% of nonstimulant cases and 29.1% of multi-class cases. Age and sex missingness were also closely related, with 92.2% of cases having unknown or missing sex also missing age. In addition, serious outcomes were more common among cases with reported age (36.7%) than cases with missing age (7.7%). These findings suggest that excluding cases with missing age could systematically change the study population.

**Overall Recommendation:** **`age` will be retained for further evaluation, with missingness handled explicitly rather than dropping incomplete cases**.

### Inspect `rq1_group` variable (Medication group)

In [ ]:
med_summary = cases["rq1_group"].value_counts().to_frame("Count")

med_summary["Percent"] = (
    cases["rq1_group"].value_counts(normalize=True) * 100
).round(1)

med_summary

,Count,Percent
rq1_group,,
STIMULANT,31154,88.6
NONSTIMULANT,3603,10.2
MULTI_ADHD_CLASS,405,1.2


There are **no missing medication groups**, which is great. However, the groups are **very unequal**.

*   ~89% are in the stimulant group
*   ~10% are in the nonstimulant group
*   ~1% is in the multiple-ADHD-class group

That's important for RQ3 because estimates for the MULTI_ADHD_CLASS group could be less stable simply because there are only 405 cases.

405 cases might still be plenty if enough of those cases experienced your serious outcome. That's something we'll check next.


#### Check serious outcomes by medication group

In [ ]:
med_outcome_summary = cases.groupby("rq1_group")["serious"].agg(
    cases="count",
    serious_cases="sum",
    serious_rate="mean"
)

med_outcome_summary["serious_percent"] = (
    med_outcome_summary["serious_rate"] * 100
).round(1)

med_outcome_summary

,cases,serious_cases,serious_rate,serious_percent
rq1_group,,,,
MULTI_ADHD_CLASS,405,116,0.286420,28.6
NONSTIMULANT,3603,1002,0.278102,27.8
STIMULANT,31154,6584,0.211337,21.1


STIMULANT makes up most of the dataset, 88.6%, but has the lowest observed serious-outcome rate at 21.1%.

NONSTIMULANT cases have a higher observed rate of 27.8%.

MULTI_ADHD_CLASS has the highest observed rate at 28.6%, although the group is much smaller at only 405 cases.

NONSTIMULANT and MULTI_ADHD_CLASS are actually quite similar in their observed rates, 27.8% versus 28.6%.

#### `rq1_group` Inspection Conclusion
Medication group was available for all 35,162 cases, with no missing values. The sample was heavily weighted toward stimulant cases: 88.6% were classified as STIMULANT, 10.2% as NONSTIMULANT, and 1.2% as MULTI_ADHD_CLASS. Serious-outcome rates differed across the groups. STIMULANT cases had the lowest observed serious-outcome rate at 21.1%, compared with 27.8% for NONSTIMULANT cases and 28.6% for MULTI_ADHD_CLASS cases. Although the **MULTI_ADHD_CLASS group** was relatively small with 405 cases, it included 116 serious outcomes, so **the group will be retained for further analysis**. These differences are descriptive and do not by themselves indicate that medication group causes differences in serious outcomes.

**Overall recommendation: KEEP the `rq1_group` variable in the analysis.**

*Note: the large difference in group size means overall model performance will be influenced heavily by the stimulant group. We'll want to look at how the model performs across groups, not just at one overall accuracy number.*

### Inspect the `wt_kg` variable

In [ ]:
cases["wt_kg"].describe()

,wt_kg
count,6714.000000
mean,71.799477
std,31.341876
min,0.000000
25%,54.583750
50%,68.440500
75%,86.000000
max,849.000000


Weight was reported for 6,714 cases (19.1%) and missing for approximately 80.9%. Among cases with reported weight, the median was 68.4 kg, with the middle 50% ranging from 54.6 to 86.0 kg. Reported values ranged from 0 to 849 kg, indicating that extreme values require further investigation before modeling.

#### High-end outliers

In [ ]:
cases.loc[
    cases["wt_kg"].notna(),
    ["wt_kg", "age_years", "age_band_fine",
     "sex_clean", "rq1_group", "serious"]
].sort_values("wt_kg", ascending=False).head(20)

,wt_kg,age_years,age_band_fine,sex_clean,rq1_group,serious
32619,849.0,13.0,12_17,M,MULTI_ADHD_CLASS,False
25408,320.0,25.0,18_25,F,STIMULANT,False
22787,300.0,28.0,26_44,M,STIMULANT,False
29244,300.0,33.0,26_44,F,STIMULANT,False
32612,290.0,37.0,26_44,M,STIMULANT,False
23798,280.0,25.0,18_25,M,STIMULANT,False
21755,260.0,30.0,26_44,M,STIMULANT,False
27980,260.0,45.0,45_64,M,STIMULANT,False
18410,250.0,58.0,45_64,M,STIMULANT,True
3657,245.0,58.0,45_64,M,STIMULANT,False


In [ ]:
cases[cases["wt_kg"] > 400][
    ["wt_kg", "age_years", "age_band_fine", "rq1_group", "serious"]
]

,wt_kg,age_years,age_band_fine,rq1_group,serious
32619,849.0,13.0,12_17,MULTI_ADHD_CLASS,False


Single case of 849kg should be treated as invalid.

In [ ]:
# Keep the original weight variable
cases["wt_kg_original"] = cases["wt_kg"]

# Create a cleaned version
cases["wt_kg_clean"] = cases["wt_kg"]

# Set clearly invalid values to missing
cases.loc[cases["wt_kg_clean"] > 400, "wt_kg_clean"] = np.nan
cases["wt_kg_clean"].describe()

,wt_kg_clean
count,6713.000000
mean,71.683702
std,29.873935
min,0.000000
25%,54.545000
50%,68.400000
75%,86.000000
max,320.000000


#### Low-end outliers

In [ ]:
cases.loc[
    cases["wt_kg"].notna(),
    ["wt_kg", "age_years", "age_band_fine",
     "sex_clean", "rq1_group", "serious"]
].sort_values("wt_kg").head(20)

,wt_kg,age_years,age_band_fine,sex_clean,rq1_group,serious
27615,0.000000,38.000000,26_44,M,STIMULANT,True
22356,0.889041,3.000000,0_5,Unknown/Missing,STIMULANT,True
24428,0.980000,0.000000,0_5,F,STIMULANT,True
25167,1.160000,0.000000,0_5,F,STIMULANT,True
24989,1.160000,0.000000,0_5,F,STIMULANT,True
24990,1.170000,0.000000,0_5,F,STIMULANT,True
25166,1.170000,0.000000,0_5,F,STIMULANT,True
25632,1.200000,NaN,None,F,STIMULANT,True
19003,1.240000,NaN,None,M,STIMULANT,False
25299,1.500000,NaN,None,F,STIMULANT,True


In [ ]:
print("Weight = 0:")
print((cases["wt_kg"] == 0).sum())

print("\nWeight > 200 kg:")
print((cases["wt_kg"] > 200).sum())

print("\nWeight > 300 kg:")
print((cases["wt_kg"] > 300).sum())

print("\nWeight > 400 kg:")
print((cases["wt_kg"] > 400).sum())

print("\nWeight < 2 kg:")
print(((cases["wt_kg"] > 0) & (cases["wt_kg"] < 2)).sum())

Weight = 0:
1

Weight > 200 kg:
32

Weight > 300 kg:
2

Weight > 400 kg:
1

Weight < 2 kg:
12


Single case of 0kg weight recorded should be treated as invalid.

In [ ]:
# Set clearly invalid values to missing
cases.loc[cases["wt_kg_clean"] == 0, "wt_kg_clean"] = np.nan
cases["wt_kg_clean"].describe()

,wt_kg_clean
count,6712.000000
mean,71.694382
std,29.863342
min,0.889041
25%,54.661250
50%,68.440500
75%,86.000000
max,320.000000


#### Serious outcome by weight reported vs. not

In [ ]:
cases["weight_missing"] = cases["wt_kg_clean"].isna()

weight_missing_summary = cases.groupby("weight_missing")["serious"].agg(
    cases="count",
    serious_cases="sum",
    serious_rate="mean"
)

weight_missing_summary["serious_percent"] = (
    weight_missing_summary["serious_rate"] * 100
).round(1)

weight_missing_summary

,cases,serious_cases,serious_rate,serious_percent
weight_missing,,,,
False,6712,1296,0.193087,19.3
True,28450,6406,0.225167,22.5


Unlike age and sex, missing weight does not show a huge difference in serious-outcome rates.

#### Investigate if weight reporting differs by variable



Does weight reporting differ substantially by medication group?

In [ ]:
pd.crosstab(
    cases["rq1_group"],
    cases["weight_missing"],
    normalize="index"
) * 100

weight_missing,False,True
rq1_group,,
MULTI_ADHD_CLASS,20.987654,79.012346
NONSTIMULANT,25.978351,74.021649
STIMULANT,18.267317,81.732683


While there is some variation, weight is missing for the majority of every medication group. Nonstimulant cases have weight reported most often at about 26%, while stimulant cases have it least often at about 18%. That matters because if we analyzed only cases with weight, the medication-group mix could shift somewhat.

Does weight reporting differ substantially by sex?

In [ ]:
pd.crosstab(
    cases["sex_clean"],
    cases["weight_missing"],
    normalize="index"
) * 100

weight_missing,False,True
sex_clean,,
F,29.446042,70.553958
M,22.177988,77.822012
Unknown/Missing,1.486200,98.513800


Among cases where sex is unknown/missing, **98.5% are also missing weight**. This is consistent with what we saw with age. **When one demographic field is missing, other demographic fields are often missing too.**

Does weight reporting differ substantially by age?

In [ ]:
pd.crosstab(
    cases["age_band_fine"],
    cases["weight_missing"],
    normalize="index"
) * 100

weight_missing,False,True
age_band_fine,,
0_5,26.666667,73.333333
12_17,23.515249,76.484751
18_25,26.867572,73.132428
26_44,34.542541,65.457459
45_64,27.477995,72.522005
65_PLUS,20.529801,79.470199
6_11,28.222836,71.777164


The 26–44 group has the highest reporting rate at 34.5%, while the 65+ group has the lowest among these known age bands at 20.5%. This only accounts for cases with a known `age_band_fine`. It doesn't show the 51% of cases with missing age. So we can't interpret this as saying weight is reasonably complete for age overall.

#### `wt_kg` Inspection Conclusion

Weight was available for 6,712 cases (19.1%) after two clearly invalid values were recoded as missing, while 80.9% of cases lacked usable weight information. Serious outcomes occurred in 19.3% of cases with reported weight and 22.5% of cases with missing weight. Weight missingness was also uneven across the study population. Weight was reported in 26.0% of NONSTIMULANT cases, 21.0% of MULTI_ADHD_CLASS cases, and 18.3% of STIMULANT cases. Missingness was especially pronounced among cases with unknown or missing sex, where 98.5% also lacked weight. Weight reporting also varied across known age groups, ranging from 20.5% among cases aged 65+ to 34.5% among cases aged 26–44.

**Overall Recommendation: Because requiring weight would exclude approximately 81% of the study population and the available-weight subset is not evenly distributed across other characteristics, weight will not be required in the primary RQ3 model**.  

---

## Combine Tables - Create a `dose_structured` variable from `adhd_drug` table & merge.
`primaryid` is the key.
`dose_structured` is a variable I'm creating. It is flagged when there is a dose amount and unit on the primary suspect ADHD drug row.

Reason for choosing the primary suspect ADHD drug row:

*   Some reports list more than one ADHD drug.
*   Reports with more drugs are more likely to have a dose written down.

To keep things fair, we count the dose from only the first ADHD drug on each report.

`adhd_cases` table has 1 row per case,  but `adhd_drug` can have several rows per case. If we directly join these tables, the same case could appear multiple times. We first need to make sure the `adhd_drug` table is one row per case.
Since the number of rows and the number of primary IDs are equal, we can join the tables!

In [ ]:
# Drop any earlier versions so re-running this cell is safe
dose_cols = ["dose_structured", "dose_structured_any", "dose_freetext",
             "n_suspect_rows", "n_suspect_rows_grp"]
cases = cases.drop(columns=[c for c in dose_cols if c in cases.columns])

dose_fix = con.sql("""
    WITH suspect AS (
        SELECT primaryid,
               CASE WHEN TRIM(CAST(dose_amt  AS VARCHAR)) <> ''
                     AND TRIM(CAST(dose_unit AS VARCHAR)) <> ''
                    THEN 1 ELSE 0 END AS has_dose,
               ROW_NUMBER() OVER (
                   PARTITION BY primaryid
                   ORDER BY CASE role_cod WHEN 'PS' THEN 0 ELSE 1 END,
                            TRY_CAST(drug_seq AS INTEGER)
               ) AS rn
        FROM adhd_drug
        WHERE role_cod IN ('PS','SS') AND cohort_medication_matched
    )
    SELECT primaryid,
           MAX(CASE WHEN rn = 1 THEN has_dose END) AS dose_structured,      -- primary suspect row only
           MAX(has_dose)                           AS dose_structured_any,  -- any row
           COUNT(*)                                AS n_suspect_rows
    FROM suspect
    GROUP BY primaryid
""").df()

print(len(dose_fix), dose_fix["primaryid"].nunique())   # should match each other

cases = cases.merge(dose_fix, on="primaryid", how="left")
print(len(cases), cases["caseid"].nunique())            # still 35162 and 35162

cases["n_suspect_rows_grp"] = cases["n_suspect_rows"].clip(upper=4)
cases["dose_structured"].value_counts(dropna=False)

94803 94803
35162 35162


,count
dose_structured,
0,24099
1,11063


The merge worked. We still have 35,162 unique cases and the `dose_structured` variable has values '0' and '1'


*   0 = No dose amount and unit reported
*   1 = Dose amount and unit reported for the primary ADHD suspect drug

**About 1 in 3 reports (31.5%) has a dose written down.**


---

## Inspect `adhd_cases` table with newly added `dose_structured`

### How often is dose reported?

In [ ]:
# 1. Serious-outcome rate by dose reporting
dose_out = cases.groupby("dose_structured")["serious"].agg(
    cases="count", serious_cases="sum", serious_rate="mean"
)
dose_out["serious_percent"] = (dose_out["serious_rate"] * 100).round(1)
print(dose_out)

                 cases  serious_cases  serious_rate  serious_percent
dose_structured                                                     
0                24099           5548      0.230217             23.0
1                11063           2154      0.194703             19.5


**Reports with a dose had slightly fewer serious outcomes (19.5%) than reports without one (23.0%).**



In [ ]:
# 2. Dose reporting by medication group
print(pd.crosstab(cases["rq1_group"], cases["dose_structured"],
                  normalize="index").round(3) * 100)

dose_structured      0     1
rq1_group                   
MULTI_ADHD_CLASS  57.5  42.5
NONSTIMULANT      70.0  30.0
STIMULANT         68.5  31.5


Stimulant and nonstimulant cases look almost the same, at about 30% each. The multi-class group is the outlier at 42.5%.

In [ ]:
# 3. Dose reporting by sex and by age missingness
print(pd.crosstab(cases["sex_clean"], cases["dose_structured"],
                  normalize="index").round(3) * 100)
print(pd.crosstab(cases["age_missing"], cases["dose_structured"],
                  normalize="index").round(3) * 100)

dose_structured     0     1
sex_clean                  
F                62.9  37.1
M                66.2  33.8
Unknown/Missing  78.9  21.1
dose_structured     0     1
age_missing                
False            65.3  34.7
True             71.7  28.3


Dose reporting is lower when sex is unknown or age is missing, so it follows the same completeness pattern as the other fields. The pattern is much weaker than it was for weight, where only 1.5% of unknown-sex cases had a weight. Even in the unknown-sex group, about 21% have a dose. So `dose_structured` seems like a meaningful variable.

### `dose_structured` Inspection Conclusion


Dose reporting was defined as whether a dose amount and unit were recorded for the first suspect ADHD drug on a report. A dose was reported for 31.5% of cases and not reported for 68.5%. Serious outcomes were slightly less common when a dose was reported (19.5%) than when it was not (23.0%). Dose reporting was lower when sex was unknown (21.1%) or age was missing (28.3%).

**Overall Recommendation: Because the variable shows real variation, it will be kept as a predictor for RQ3.**

---

## Combine Tables - Create a `route_group` variable from `adhd_drug` table & merge.

Route is typed in by whoever filed the report, so it is messy. For example, "ORAL", "Oral" and "by mouth" may all appear. This query counts the values for the *first suspect ADHD drug row* on each report, the same row we used for `dose_structured`, because a report can list more than one drug. It cleans up capitalization and spaces.


**Definition: First Suspect ADHD**

*   **Suspect.** The reporter thinks the drug may have caused the problem. PS is the primary suspect, SS is a secondary suspect, and C is a concomitant drug that was only taken at the same time. We keep only PS and SS rows.
*  **ADHD**. The drug is one of the ADHD medicines in your study, confirmed by the cohort_medication_matched check. Rows like *Risperidone* or *Ibuprofen* are dropped.
*   **First.** If a report still has more than one row, the primary suspect goes first. If there's no primary suspect, the row with the lowest drug_seq (*the order the drugs were listed*) goes first.

*Note: 81% of cases have only one suspect ADHD row, so for them there is nothing to choose.*







In [ ]:
# One row per report: the first suspect ADHD drug (primary suspect first, then lowest drug_seq)
first_drug = con.sql("""
    SELECT primaryid, route
    FROM adhd_drug
    WHERE role_cod IN ('PS','SS') AND cohort_medication_matched
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY primaryid
        ORDER BY CASE role_cod WHEN 'PS' THEN 0 ELSE 1 END,
                 TRY_CAST(drug_seq AS INTEGER)
    ) = 1
""").df()

print(len(first_drug), first_drug["primaryid"].nunique())   # should match each other

94803 94803


In [ ]:
# Capitalize, trim spaces, and label blanks so spelling variations match
first_drug["route_value"] = (
    first_drug["route"].fillna("").astype(str).str.strip().str.upper().replace("", "(BLANK)")
)

route_look = first_drug["route_value"].value_counts().to_frame("reports")
route_look

,reports
route_value,
ORAL,32920
UNKNOWN,29562
(BLANK),20277
TRANSDERMAL,9933
TOPICAL,623
ORAL USE,466
TRANSPLACENTAL,424
INTRAVENOUS (NOT OTHERWISE SPECIFIED),260
NASAL,79


### Group the `route_value` into `route_group` & merge
Groups are Oral, Transdermal, Unknown/Missing or Other.


In [ ]:
# Define grouping rules. Any route not listed here becomes "Other"
route_groups = {
    "ORAL": "Oral", "ORAL USE": "Oral", "INGESTION": "Oral",
    "TRANSDERMAL": "Transdermal", "TRANSDERMAL USE": "Transdermal",
    "UNKNOWN": "Unknown/Missing", "(BLANK)": "Unknown/Missing",
}

first_drug["route_group"] = first_drug["route_value"].map(route_groups).fillna("Other")

In [ ]:
# Did grouping work?
check = first_drug["route_group"].value_counts().to_frame("reports")
check["percent"] = (check["reports"] / check["reports"].sum() * 100).round(1)
print(check)
print("Every report has a group:", check["reports"].sum() == len(first_drug))

# What was grouped into "Other"?
first_drug.loc[first_drug["route_group"] == "Other", "route_value"].value_counts()

                 reports  percent
route_group                      
Unknown/Missing    49839     52.6
Oral               33388     35.2
Transdermal         9935     10.5
Other               1641      1.7
Every report has a group: True


,count
route_value,
TOPICAL,623
TRANSPLACENTAL,424
INTRAVENOUS (NOT OTHERWISE SPECIFIED),260
NASAL,79
OTHER,63
PARENTERAL,47
INTRAVENOUS,31
RESPIRATORY (INHALATION),30
ENDOCERVICAL,11


In [ ]:
# For rerun purposes: removes an earlier route_group before merging
cases = cases.drop(columns=[c for c in ["route_group"] if c in cases.columns])

cases = cases.merge(first_drug[["primaryid", "route_group"]], on="primaryid", how="left")
print(len(cases), cases["caseid"].nunique())   # still 35162 and 35162

35162 35162


In [ ]:
# Look at the counts
route_summary = cases["route_group"].value_counts(dropna=False).to_frame("Count")
route_summary["Percent"] = (
    cases["route_group"].value_counts(dropna=False, normalize=True) * 100
).round(1)
route_summary

,Count,Percent
route_group,,
Unknown/Missing,23646,67.2
Oral,9384,26.7
Transdermal,1203,3.4
Other,929,2.6


About two-thirds of cases do not have a known route of administration. Oral is the only large known group. Transdermal and Other are small but big enough to compare (~1,000 cases each).

---

## Inspect `adhd_cases` table with newly added `route_group`

#### Serious outcome rate by `route_group`

In [ ]:
route_outcome = cases.groupby("route_group")["serious"].agg(
    cases="count", serious_cases="sum", serious_rate="mean"
)
route_outcome["serious_percent"] = (route_outcome["serious_rate"] * 100).round(1)
route_outcome.sort_values("cases", ascending=False)

,cases,serious_cases,serious_rate,serious_percent
route_group,,,,
Unknown/Missing,23646,4532,0.191660,19.2
Oral,9384,2866,0.305413,30.5
Transdermal,1203,17,0.014131,1.4
Other,929,287,0.308934,30.9


For Transdermal route, only 17 of 1,203 cases were serious, far below every other group. Oral and Other are almost the same (30.5% vs. 30.9%).
Unknown/Missing is lower than the known routes (19.2%). Missing sex and missing age showed the same pattern: reports with less information have fewer serious outcomes.

#### Investigate if certain groups are more likely to report route

In [ ]:
print(pd.crosstab(cases["sex_clean"], cases["route_group"], normalize="index").round(3) * 100)
print(pd.crosstab(cases["rq1_group"], cases["route_group"], normalize="index").round(3) * 100)

route_group      Oral  Other  Transdermal  Unknown/Missing
sex_clean                                                 
F                32.4    2.1          3.4             62.1
M                29.2    2.9          5.9             62.0
Unknown/Missing  16.0    3.0          0.3             80.6
route_group       Oral  Other  Transdermal  Unknown/Missing
rq1_group                                                  
MULTI_ADHD_CLASS  29.6    2.0          0.7             67.7
NONSTIMULANT      28.4    2.6          0.2             68.8
STIMULANT         26.4    2.7          3.8             67.1


**Sex:** Route is missing more often when sex is unknown (80.6% vs. about 62%). That is the same completeness pattern as before.

**Medication group:** Route is missing about two-thirds of the time in every group, so missingness isn't tied to group. Transdermal is the exception. It is almost entirely stimulant cases, because 3.8% of stimulant cases is roughly 1,200 cases, which is nearly all 1,203 transdermal cases. That means the transdermal group and the stimulant group overlap heavily.

### `route_group` Inspection Conclusion

Route was taken from the first suspect ADHD drug row on each report and grouped as Oral, Transdermal, Other, or Unknown/Missing. Route was unknown or missing for 67.2% of cases, Oral for 26.7%, Transdermal for 3.4%, and Other for 2.6%.

Serious outcomes were most common among Other (30.9%) and Oral (30.5%) routes, less common when route was unknown or missing (19.2%), and rare for the Transdermal route (1.4%, 17 of 1,203 cases).

Route was more often missing when sex was unknown (80.6%) than when sex was F or M (about 62%).

Transdermal cases were almost entirely stimulant cases.

**Overall Recommendation: Because route shows clear differences in serious-outcome rates, it will be retained in the primary RQ3 model. These differences are descriptive and do not indicate that route causes differences in outcomes.**

---

## Combine Tables - Create an `n_other_meds` variable from `adhd_drug` table & merge.

`primaryid` is the key.

`n_other_meds` is the number of other (non-ADHD study) drugs listed on each report. These are the 'concurrent medications' in RQ3. We're asking: how many other medicines does each report list, besides the ADHD study drug?

`adhd_drug` has one row per drug, so we count the non-ADHD rows for each report first, then join. This keeps one row per case.

Reports that list no other drugs will have no rows to count, so their count is set to 0.

*Note: The same drug can be listed twice under a brand and a generic name, so the count may run slightly high.*

### Count how many cases have other drugs listed

In [ ]:
other_meds = con.sql("""
    SELECT primaryid,
           COUNT(DISTINCT drug_seq) AS n_other_meds
    FROM adhd_drug
    WHERE COALESCE(cohort_medication_matched, false) = false -- keeps rows with blank or false
    GROUP BY primaryid
""").df()

print(len(other_meds), other_meds["primaryid"].nunique())   # should match each other

34664 34664


### Merge into `adhd_cases` and fill in zeros

In [ ]:
# Need for re-run: removes earlier versions before merging
cases = cases.drop(columns=[c for c in ["n_other_meds", "other_meds_group"] if c in cases.columns])

cases = cases.merge(other_meds, on="primaryid", how="left")
cases["n_other_meds"] = cases["n_other_meds"].fillna(0).astype(int)   # no rows found = 0 other drugs

print(len(cases), cases["caseid"].nunique())   # still 35162 and 35162

35162 35162


### Inspect the distribution of values for `n_other_meds`

In [ ]:
print(cases["n_other_meds"].describe())
cases["n_other_meds"].value_counts().sort_index().head(15)

count    35162.000000
mean         4.647403
std         32.203234
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max       1456.000000
Name: n_other_meds, dtype: float64


,count
n_other_meds,
0,21091
1,3834
2,2565
3,1824
4,1173
5,862
6,633
7,541
8,406


Most cases (60.0%, 21,091) list no other drugs besides the ADHD study drug. The distribution has a long tail: the mean number of other drugs is 4.6, the median is 0, and the maximum is 1,456. To keep extreme reports from distorting results, counts will be grouped as 0, 1-2, 3-5, and 6+, with each group holding at least 10% of cases.

### Bin the counts

In [ ]:
bins   = [-1, 0, 2, 5, float("inf")]
labels = ["0", "1-2", "3-5", "6+"]

cases["other_meds_group"] = pd.cut(cases["n_other_meds"], bins=bins, labels=labels)

In [ ]:
check = cases["other_meds_group"].value_counts(dropna=False).sort_index().to_frame("cases")
check["percent"] = (check["cases"] / check["cases"].sum() * 100).round(1)
print(check)
print("Every case has a group:", check["cases"].sum() == len(cases))

                  cases  percent
other_meds_group                
0                 21091     60.0
1-2                6399     18.2
3-5                3859     11.0
6+                 3813     10.8
Every case has a group: True


### Inspect the counts by group

In [ ]:
other_summary = cases["other_meds_group"].value_counts(dropna=False).sort_index().to_frame("Count")
other_summary["Percent"] = (
    cases["other_meds_group"].value_counts(dropna=False, normalize=True).sort_index() * 100
).round(1)
other_summary

,Count,Percent
other_meds_group,,
0,21091,60.0
1-2,6399,18.2
3-5,3859,11.0
6+,3813,10.8


---

## Inspect `adhd_cases` table with newly added `other_meds_group`

#### Serious outcome rate by `other_meds_group`

In [ ]:
other_outcome = cases.groupby("other_meds_group", observed=True)["serious"].agg(
    cases="count", serious_cases="sum", serious_rate="mean"
)
other_outcome["serious_percent"] = (other_outcome["serious_rate"] * 100).round(1)
other_outcome

,cases,serious_cases,serious_rate,serious_percent
other_meds_group,,,,
0,21091,1876,0.088948,8.9
1-2,6399,1928,0.301297,30.1
3-5,3859,1706,0.442083,44.2
6+,3813,2192,0.574875,57.5


As the number of other meds increase, so do the serious cases. This is a strong pattern.

#### Investigate if certain groups are more likely to list other medicines

In [ ]:
print(pd.crosstab(cases["sex_clean"], cases["other_meds_group"], normalize="index").round(3) * 100)
print(pd.crosstab(cases["rq1_group"], cases["other_meds_group"], normalize="index").round(3) * 100)

other_meds_group     0   1-2   3-5    6+
sex_clean                               
F                 49.0  21.9  14.3  14.7
M                 52.2  22.1  13.4  12.3
Unknown/Missing   84.1   8.5   3.6   3.8
other_meds_group     0   1-2   3-5    6+
rq1_group                               
MULTI_ADHD_CLASS  38.0  24.2  15.3  22.5
NONSTIMULANT      52.3  18.0  10.4  19.2
STIMULANT         61.2  18.1  11.0   9.7


We are noticing the same completeness pattern that we have seen for the other variables. Sparse reports leave most other variables blank.

Stimulant reports list the fewest other drugs, with 61.2% at 0 and 9.7% at 6+. Nonstimulant and multi-class reports list more, and 22.5% of multi-class cases have 6 or more drugs listed.

Another important note - the '0' group has the fewest serious cases reported (8.9%), but it also contains many cases with missing/unknown data (84% missing/unknown sex), and unknown-sex cases had a low serious rate (8.3%). Part of the low rate of serious cases in this group may therefore reflect how little was reported, not the absence of other medicines. In addition, more other medicines going with more serious outcomes doesn't mean the other medicines caused them.

### `n_other_meds` Inspection Conclusion

The number of other (non-ADHD) drugs listed on each report was grouped as 0 (60.0%), 1-2 (18.2%), 3-5 (11.0%), and 6+ (10.8%).

The serious-outcome rate rose with each group: 8.9% for 0, 30.1% for 1-2, 44.2% for 3-5, and 57.5% for 6+.

Reports with unknown or missing sex mostly listed no other drugs (84.1%), compared with about 50% for F and M.

Stimulant cases listed the fewest other drugs (61.2% had none), while 22.5% of MULTI_ADHD_CLASS cases listed six or more.

**Overall Recommendation: Because the serious-outcome rate differs markedly across groups, the variable will be retained in the primary RQ3 model. These differences are descriptive and do not indicate that other medicines cause serious outcomes.**

---

## Inspect case counts and serious-outcome rate (train-test split)
Check whether we have enough cases, anda similar serious-outcome rate, in the training period (2022 - 2024) and test period (2025 - Feb 2026).

In [ ]:
# What does init_month look like, and how many are missing?
print(cases["init_month"].dtype)
print(cases["init_month"].isna().sum(), "cases with no date")
print(cases["init_month"].min(), "to", cases["init_month"].max())
cases["init_month"].head()

datetime64[us]
0 cases with no date
2007-04-01 00:00:00 to 2026-06-01 00:00:00


,init_month
0,2014-03-01
1,2014-03-01
2,2014-04-01
3,2014-07-01
4,2014-08-01


### Count cases by year & Group by Serious Rate and Percent

In [ ]:
cases["init_date"] = pd.to_datetime(cases["init_month"])
cases["init_year"] = cases["init_date"].dt.year

year_summary = cases.groupby("init_year")["serious"].agg(
    cases="count", serious_cases="sum", serious_rate="mean"
)
year_summary["serious_percent"] = (year_summary["serious_rate"] * 100).round(1)
year_summary

,cases,serious_cases,serious_rate,serious_percent
init_year,,,,
2007,1,1,1.000000,100.0
2008,2,2,1.000000,100.0
2010,6,3,0.500000,50.0
2011,3,2,0.666667,66.7
2012,1,0,0.000000,0.0
2013,2,0,0.000000,0.0
2014,9,4,0.444444,44.4
2015,29,7,0.241379,24.1
2016,52,18,0.346154,34.6


### Sort into train-test-split periods

In [ ]:
# Safe to rerun
cases = cases.drop(columns=[c for c in ["split"] if c in cases.columns])

def assign_split(d):
    if d < pd.Timestamp("2022-01-01"):
        return "Before 2022"
    if d < pd.Timestamp("2025-01-01"):
        return "Train (2022-2024)"
    if d < pd.Timestamp("2026-03-01"):
        return "Test (Jan 2025-Feb 2026)"
    return "After Feb 2026"

cases["split"] = cases["init_date"].apply(assign_split)

split_summary = cases.groupby("split")["serious"].agg(
    cases="count", serious_cases="sum", serious_rate="mean"
)
split_summary["serious_percent"] = (split_summary["serious_rate"] * 100).round(1)
split_summary["percent_of_cases"] = (split_summary["cases"] / len(cases) * 100).round(1)
split_summary

,cases,serious_cases,serious_rate,serious_percent,percent_of_cases
split,,,,,
After Feb 2026,2751,456,0.165758,16.6,7.8
Before 2022,996,333,0.334337,33.4,2.8
Test (Jan 2025-Feb 2026),9356,2342,0.250321,25.0,26.6
Train (2022-2024),22059,4571,0.207217,20.7,62.7


Cases were placed on the timeline using the month the FDA first received them.

No cases were missing a date.

Of 35,162 cases, 22,059 (62.7%) fell in the training period (2022-2024) and 9,356 (26.6%) in the test period (January 2025-February 2026).

The serious-outcome rate was 20.7% in the training period and 25.0% in the test period.

A further 996 cases (2.8%) were first received before 2022 and 2,751 (7.8%) after February 2026. These 3,747 cases fall outside both periods and will be excluded from model training and testing, leaving **31,415 cases for modeling.**